In [1]:
import time
import numpy as np
import json
from tqdm import tqdm
import uuid

import pandas as pd

from stanza_tokenizer import StanzaTokenizer

/Users/stevie/repos/lingo_kit_combined/lingo_kit_data/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-10-20 12:40:48 INFO: Downloaded file to /Users/stevie/stanza_resources/resources.json
2025-10-20 12:40:48 INFO: Downloading default packages for language: it (Italian) ...
2025-10-20 12:40:49 INFO: File exists: /Users/stevie/stanza_resources/it/default.zip
2025-10-20 12:40:50 INFO: Finished downloading models and saved to /Users/stevie/stanza_resources


In [2]:
tokenizer = StanzaTokenizer()

2025-10-20 12:40:50 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2025-10-20 12:40:51 INFO: Downloaded file to /Users/stevie/stanza_resources/resources.json
2025-10-20 12:40:51 WARNING: Language it package default expects mwt, which has been added
2025-10-20 12:40:51 INFO: Loading these models for language: it (Italian):
| Processor | Package           |
---------------------------------
| tokenize  | combined          |
| mwt       | combined          |
| pos       | combined_charlm   |
| lemma     | combined_nocharlm |
| depparse  | combined_charlm   |

2025-10-20 12:40:51 INFO: Using device: cpu
2025-10-20 12:40:51 INFO: Loading: tokenize
2025-10-20 12:40:52 INFO: Loading: mwt
2025-10-20 12:40:52 INFO: Loading: pos
2025-10-20 12:40:53 INFO: Loading: lemma
2025-10-20 12:40:53 INFO: Loading: depparse
2025-10-20 12:40:53 INFO: Done loading 

In [3]:
path = '/Users/stevie/repos/lingo_kit_combined/lingo_kit_data/word_analysis/datasets/tatoeba/dataframe.tsv'
df = pd.read_csv(path, sep='\t')
len(df), df.columns

(624335, Index(['text_it', 'text_en', 'hash'], dtype='object'))

In [4]:
df.head()

,text_it,text_en,hash
0,Devo andare a dormire.,I need to go to sleep.,ec6eb055-76b1-5399-81ba-5ad8476457ad
1,Che cos'è?,What is it?,07d00a09-108d-5cc2-927c-51751a2cd16f
2,La parola d'accesso è Muiriel.,The password is Muiriel.,784812fc-91aa-5bc7-874f-e01e64df6328
3,Non cambierà niente.,That won't change anything.,d8fbd928-cb68-532b-aadc-1bd194ac0482
4,Costerà trenta euro.,This will cost €30.,beff8df5-d1e5-50cf-a9da-e7698e99baba


In [5]:
def get_token_hash(term, lemma, pos):
    return str(uuid.uuid5(uuid.NAMESPACE_DNS, f"{term}-{lemma}-{pos}"))

def get_group_hash(lemma, pos):
    return str(uuid.uuid5(uuid.NAMESPACE_DNS, f"{lemma}-{pos}"))

In [6]:
def normalize(text):
    text = text.lower()
    text = text.replace("’", "'")
    text = text.replace("“", '"').replace("”", '"')
    text = text.strip()
    return text

In [ ]:
start_i = 180000
end_i =  200000
df = df.iloc[start_i:end_i]
print(f"Processing rows {start_i} to {end_i}")
print(f"Total rows: {len(df)}")

Processing rows 160000 to 180000
Total rows: 20000


In [8]:
print(df.head(n=5))
print(df[0:5])

                                                text_it  \
160000  Tom ha detto che Mary era una brava nuotatrice.   
160001     Tom disse che Mary era una brava nuotatrice.   
160002                    Non ha nessuno da consultare.   
160003                Lui non ha nessuno da consultare.   
160004                  Tom ha passato la palla a Mary.   

                                  text_en  \
160000  Tom said Mary was a good swimmer.   
160001  Tom said Mary was a good swimmer.   
160002          He has nobody to consult.   
160003          He has nobody to consult.   
160004       Tom passed the ball to Mary.   

                                        hash  
160000  47a91883-00ba-5e8e-9075-fa6b02d2bf4d  
160001  fcdd72b5-1bfe-5751-bbfd-b29b6e5f0c92  
160002  d8bfaa3c-2f9b-5b45-a34c-c7d34e76cc6f  
160003  ae4e850e-6f0e-5de8-85fd-0cb1f80c7af8  
160004  fb63a922-b3e5-541d-9f93-e38332a7b942  
                                                text_it  \
160000  Tom ha detto che Mary era

In [9]:
data = {}
for _, row in tqdm(df.iterrows(), total=len(df)):
    tokens = tokenizer.tokenize(row['text_it'])
    for token in tokens:
        word = normalize(token['text'])
        token_hash = get_token_hash(word, token['lemma'], token['pos'])
        group_hash = get_group_hash(token['lemma'], token['pos'])
        if token_hash not in data:
            data[token_hash] = {
                'word': word,
                'lemma': token['lemma'],
                'pos': token['pos'],
                'xpos': token['xpos'],
                'deprel': token['deprel'],
                'count': 0,
                'sentences': set(),
                'group_hash': group_hash,
            }
        data[token_hash]['count'] += 1
        data[token_hash]['sentences'].add(row['hash'])

100%|██████████| 20000/20000 [18:12<00:00, 18.30it/s]  


In [10]:
token_df_data = {'token_hash': [], 'word': [], 'lemma': [], 'pos': [], 'xpos': [], 'deprel': [], 'count': [], 'sentences': [], 'group_hash': []}
for token_hash, info in data.items():
    if info['pos'] == 'PUNCT':
        continue
    token_df_data['token_hash'].append(token_hash)
    token_df_data['word'].append(info['word'])
    token_df_data['lemma'].append(info['lemma'])
    token_df_data['pos'].append(info['pos'])
    token_df_data['xpos'].append(info['xpos'])
    token_df_data['deprel'].append(info['deprel'])
    token_df_data['count'].append(info['count'])
    token_df_data['sentences'].append(list(info['sentences']))
    token_df_data['group_hash'].append(info['group_hash'])
token_df = pd.DataFrame(token_df_data)

In [11]:
token_df.sort_values(by='count', ascending=False, inplace=True)

In [12]:
save_path = f'/Users/stevie/repos/lingo_kit_combined/lingo_kit_data/word_analysis/datasets/tatoeba/token_data/token_data_{start_i}_{end_i}.tsv'
token_df.to_csv(save_path, sep='\t', index=False)

In [13]:
token_df = pd.read_csv(save_path, sep='\t')
len(token_df), token_df.columns

(8299,
 Index(['token_hash', 'word', 'lemma', 'pos', 'xpos', 'deprel', 'count',
        'sentences', 'group_hash'],
       dtype='object'))